# Cross-Dataset Phishing Website Detection
## Notebook 3: In-Dataset Baseline Modeling (Experiments 1 & 2)

**Core Concept:**
Before evaluating across datasets, we establish performance baselines by training and testing within the same distribution.
We evaluate 5 representative model families:
1. **Logistic Regression** (Linear baseline)
2. **Decision Tree** (Non-parametric tree)
3. **Random Forest** (Bagging ensemble)
4. **XGBoost** (Gradient boosting ensemble)
5. **Support Vector Machine (SVM)** (Margin-based classifier)

**Data Leakage Prevention:**
All transformations (scaling, sigmoid calibration) are encapsulated within `sklearn.pipeline.Pipeline`, fitting parameters exclusively on training folds.


In [ ]:
# Environment & Path Setup (supports local execution and Google Colab)
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/phishing-project')
except ImportError:
    BASE = Path('..').resolve() if Path('.').resolve().name == 'notebooks' else Path('.').resolve()

sys.path.insert(0, str(BASE))
print(f"Project base path: {BASE}")

from src.data_utils import project_paths
dirs = project_paths(BASE)
PROC_DIR = dirs['processed']
TAB_DIR = dirs['tables']
MOD_DIR = dirs['models']
FIG_DIR = dirs['figures']


### 1. Load Processed Datasets
We load both the full feature datasets and the 9-feature aligned datasets.


In [ ]:
import pandas as pd
from src.data_utils import load_processed
from src.evaluation import split_data

# Full feature datasets
phi_clean, uci_clean, meta = load_processed(PROC_DIR)

# Aligned datasets (9 features)
phi_al = pd.read_csv(PROC_DIR / 'phiusiil_aligned.csv')
uci_al = pd.read_csv(PROC_DIR / 'uci_aligned.csv')

print(f"PhiUSIIL Clean: {phi_clean.shape} | Aligned: {phi_al.shape}")
print(f"UCI Clean:      {uci_clean.shape} | Aligned: {uci_al.shape}")


### 2. Inspect Baseline Model Pipelines
Models are defined in `src.models.get_model_dict`. Each estimator is paired with proper preprocessing:
- Distance-based models (LR, SVM) receive `StandardScaler`.
- Tree-based models (DT, RF, XGB) receive `passthrough` to preserve natural splits.


In [ ]:
from src.models import get_model_dict

models_phi = get_model_dict(dataset_size="large")
models_uci = get_model_dict(dataset_size="small")

print("Available Model Architectures:")
for name in models_phi.keys():
    print(f" - {name}")


### 3. Load & Display In-Dataset Baseline Results
All models were trained using Stratified 80/20 splits with 5-Fold Cross-Validation.


In [ ]:
res_df = pd.read_csv(TAB_DIR / '03_baseline_results.csv')
display(res_df[['experiment', 'model', 'test_accuracy', 'test_precision', 'test_recall', 'test_f1', 'test_roc_auc']])


### 4. Overlaid ROC Curves
Visualizing discrimination thresholds across all 5 models for the aligned feature subsets.


In [ ]:
from IPython.display import Image, display

print("Experiment 1: PhiUSIIL (Aligned Features) ROC Curves")
display(Image(filename=str(FIG_DIR / 'roc_Exp1_aligned.png')))

print("Experiment 2: UCI Phishing (Aligned Features) ROC Curves")
display(Image(filename=str(FIG_DIR / 'roc_Exp2_aligned.png')))


### 5. In-Dataset Analysis & Key Observations
1. **PhiUSIIL (All Features):** Achieves near 100% accuracy across all models. As investigated in Notebook 04, features like `URLSimilarityIndex` provide deterministic separation.
2. **PhiUSIIL (Aligned 9 Features):** Realistic performance ($F_1 = 0.828 - 0.863$). Tree models prioritize Precision (0.973), while Linear models prioritize Recall (0.945).
3. **UCI Phishing (Aligned 9 Features):** Ensembles (Random Forest and XGBoost) achieve the highest $F_1$ scores (~0.881) and ROC-AUC (~0.934).
